# EDA – CHRONO24 PRICE PREDICTION MODELS

# Load Data and Inspect

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv("/kaggle/input/datasets/philmorekoung11/luxury-watch-listings/Watches.csv")

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nMissing values (%):\n{(df.isnull().sum() / len(df) * 100).round(2)}")
print(f"\nFull duplicates: {df.duplicated().sum()}")
df.head()

In [ ]:
for col in df.columns:
    print(f"\n--- {col} ---")
    print(df[col].dropna().unique()[:10])

# Cleaning

In [ ]:
df = df.drop(columns=['Unnamed: 0'])

df['price'] = df['price'].str.replace(r'[\$,]', '', regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]

df['condition'] = df['condition'].fillna(df['cond'])
df = df.drop(columns=['cond'])

df['yop_approx'] = df['yop'].str.contains('Approximation', na=False).astype(int)
df['yop'] = pd.to_numeric(df['yop'].str.extract(r'(\d{4})')[0], errors='coerce')

df['size_mm'] = pd.to_numeric(df['size'].str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
df = df.drop(columns=['size'])

for col in ['mvmt', 'casem', 'bracem', 'sex', 'condition', 'model', 'brand']:
    df[col] = df[col].fillna('Unknown')

df = df.drop_duplicates(subset=['name', 'ref', 'price', 'brand'], keep='first')

print(f"Shape after cleaning: {df.shape}")
print(f"\nMissing values (%):\n{(df.isnull().sum() / len(df) * 100).round(2)}")
print(f"\nPrice stats:\n{df['price'].describe()}")
print(f"\nyop stats:\n{df['yop'].describe()}")
print(f"\nsize_mm stats:\n{df['size_mm'].describe()}")

# EDA + Anomaly Treatment

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df.loc[(df['size_mm'] < 15) | (df['size_mm'] > 60), 'size_mm'] = np.nan
df.loc[(df['yop'] < 1900) | (df['yop'] > 2026), 'yop'] = np.nan

low, high = df['price'].quantile([0.005, 0.995])
df = df[(df['price'] >= low) & (df['price'] <= high)]

df['log_price'] = np.log1p(df['price'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(df['price'], bins=60, ax=axes[0, 0])
axes[0, 0].set_title('Price')
sns.histplot(df['log_price'], bins=60, ax=axes[0, 1])
axes[0, 1].set_title('Log Price')
sns.histplot(df['yop'].dropna(), bins=50, ax=axes[1, 0])
axes[1, 0].set_title('Year of Production')
sns.histplot(df['size_mm'].dropna(), bins=45, ax=axes[1, 1])
axes[1, 1].set_title('Size (mm)')
plt.tight_layout()
plt.show()

top_brands = df.groupby('brand')['price'].median().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
top_brands.plot(kind='bar')
plt.title('Median Price by Brand (Top 15)')
plt.ylabel('Median Price')
plt.show()

print(f"Shape: {df.shape}")
print(f"Price range: {df['price'].min():.0f} - {df['price'].max():.0f}")
print(f"\nBrand counts (top 10):\n{df['brand'].value_counts().head(50)}")
print(f"\nCorrelation with log_price:\n{df[['log_price', 'yop', 'size_mm', 'yop_approx']].corr()['log_price']}")

In [ ]:
group_map = {
    'Rolex': 'C_rolex_style',
    'Tudor': 'C_rolex_style',
    'Omega': 'B_omega_pic',
    'Audemars Piguet': 'A_ap_style',
    'Patek Philippe': 'D_patek_style',
}

df['ref_group'] = df['brand'].map(group_map).fillna('E_F_generic_other')

print(df['ref_group'].value_counts())
print(f"\nTotal watches: {len(df)}")
print(f"\nBreakdown %:\n{(df['ref_group'].value_counts(normalize=True) * 100).round(2)}")

# Feature Engineering

In [ ]:
CURRENT_YEAR = 2026

df['watch_age'] = CURRENT_YEAR - df['yop']
df['yop_missing'] = df['yop'].isna().astype(int)
df['size_missing'] = df['size_mm'].isna().astype(int)
df['yop'] = df['yop'].fillna(df['yop'].median())
df['size_mm'] = df['size_mm'].fillna(df.groupby('sex')['size_mm'].transform('median'))
df['watch_age'] = df['watch_age'].fillna(df['watch_age'].median())

df['same_material'] = (df['casem'] == df['bracem']).astype(int)
gold_pattern = 'gold|platinum'
df['precious_case'] = df['casem'].str.contains(gold_pattern, case=False, na=False).astype(int)
df['precious_brace'] = df['bracem'].str.contains(gold_pattern, case=False, na=False).astype(int)

df['name_len'] = df['name'].fillna('').str.len()
df['name_missing'] = df['name'].isna().astype(int)
df['has_limited'] = df['name'].fillna('').str.contains('limited|anniversary|edition', case=False).astype(int)

rare_brands = df['brand'].value_counts()[df['brand'].value_counts() < 10].index
df['brand'] = df['brand'].replace(rare_brands, 'Other')

model_freq = df['model'].value_counts(normalize=True)
df['model_freq'] = df['model'].map(model_freq)

df_encoded = pd.get_dummies(df, columns=['brand', 'mvmt', 'casem', 'bracem', 'sex', 'condition'],
                            drop_first=True, dtype=int)

drop_cols = ['name', 'model', 'ref', 'price']
X = df_encoded.drop(columns=drop_cols + ['log_price'])
y = df_encoded['log_price']

print(f"X shape: {X.shape}")
print(f"\nFeature list:\n{X.columns.tolist()}")
print(f"\nAny NaN left: {X.isnull().sum().sum()}")
print(f"\nTop correlations with log_price:\n{df_encoded[X.select_dtypes(include=np.number).columns.tolist() + ['log_price']].corr()['log_price'].abs().sort_values(ascending=False).head(15)}")